# 02 — Build chip datasets

Chips every city into 256×256 patches, writes memory-mapped uint16 arrays per split, computes fixed train-split normalization stats, and pushes one zip per split plus `manifest.json` to Drive. Run **once** — the outputs are permanent on Drive.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q -e .


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.data.build import build

oscd_root = store.extract_oscd()
manifest = build(oscd_root, "configs/data.yaml", drive=store,
                 scratch=store.scratch)


In [ ]:
import json
print(json.dumps({k: v for k, v in manifest.items() if k != "stats"}, indent=1))
print("mean:", [round(m, 4) for m in manifest["stats"]["mean"]])
print("std :", [round(s, 4) for s in manifest["stats"]["std"]])
